In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Ptc.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 1000,2018-03-01 10:55:17,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 1000,2018-03-01 10:55:21,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,request for payment 1000,2018-03-01 11:34:16,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,2335.0
3,request for payment 1000,2018-03-01 11:34:23,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,7.0
4,request for payment 1000,2018-03-01 15:01:48,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,12445.0
5,request for payment 1000,2018-03-05 14:49:53,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,344885.0
6,request for payment 1000,2018-03-06 10:13:29,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request Payment,SYSTEM,UNDEFINED,69816.0
7,request for payment 1000,2018-03-08 17:31:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Payment Handled,SYSTEM,UNDEFINED,199051.0
8,request for payment 10043,2018-02-20 13:53:11,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
9,request for payment 10043,2018-02-20 13:53:14,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [6.00, 362277.00]                        61266.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [64.68, 1661.05]                         338.4588   quantile_derived    
case:Permit RequestedBudget    continuous     case     yes    [130.85, 4066.04]                        769

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Payment Handled', 'Request Payment'}]

In [13]:
engine.branching_sets

[{'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTED by PRE_APPROVER',
  'Permit REJECTED by SUPERVISOR',
  'Permit SUBMITTED by EMPLOYEE',
  'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR',
  'Request For Payment FINAL_APPROVED by SUPERVISOR',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by BUDGET OWNER',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED 

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Ptc-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 70592,4,1,0,0.488408,0.501815,0.475000,0.575000,0.027273,...,0.100786,0.000000,0.000000,0.000000,0.000000,0.0000,0.100786,0.100786,0.000000,0.000000
1,0,request for payment 29593,4,1,0,0.411908,0.332149,0.491667,0.570000,0.172727,...,0.283335,0.000000,0.083335,0.166667,0.000004,0.2000,0.000000,0.000000,0.000000,0.000000
2,0,request for payment 52969,6,1,0,0.448592,0.338851,0.558333,0.540000,0.386667,...,0.416680,0.133333,0.083347,0.166667,0.000027,0.2000,0.000000,0.000000,0.000000,0.000000
3,0,request for payment 55558,7,1,0,0.406144,0.320621,0.491667,0.560000,0.470588,...,0.855122,0.470588,0.084533,0.166667,0.002400,0.3000,0.000000,0.000000,0.000000,0.000000
4,0,request for payment 45517,8,1,2,0.397697,0.378727,0.416667,0.495000,0.486842,...,0.668042,0.157895,0.000000,0.000000,0.000000,0.0000,0.510147,0.510147,0.999965,0.999965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,18,request for payment 77073,8,1,2,0.435661,0.396322,0.475000,0.596875,0.000000,...,0.248628,0.000000,0.061128,0.100000,0.022255,0.1875,0.000000,0.000000,0.000000,0.000000
162,18,request for payment 71978,8,1,2,0.368763,0.317526,0.420000,0.481250,0.000000,...,0.099138,0.000000,0.036638,0.000000,0.073275,0.0625,0.000000,0.000000,0.000000,0.000000
163,18,request for payment 82927,8,1,2,0.449140,0.383281,0.515000,0.568750,0.105263,...,0.280265,0.105263,0.050001,0.100000,0.000003,0.1250,0.000000,0.923702,0.000000,1.000000
164,18,request for payment 82310,8,1,2,0.457567,0.450134,0.465000,0.568750,0.000000,...,0.143797,0.000000,0.081297,0.000000,0.162594,0.0625,0.000000,0.000000,0.000000,0.000000


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/450 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 70592,4,2,0,0.546872,0.406245,0.687500,0.685714,0.160714,...,0.657028,0.142857,0.228456,0.250000,0.206912,0.285714,0.000000,0.370666,0.000000,0.500000
1,0,request for payment 29593,4,2,0,0.492764,0.348028,0.637500,0.585714,0.142857,...,0.678571,0.142857,0.250000,0.500000,0.000000,0.285714,0.000000,0.376365,0.000000,0.500000
2,0,request for payment 52969,6,2,0,0.385794,0.321588,0.450000,0.442857,0.358333,...,0.869048,0.333333,0.250000,0.500000,0.000000,0.285714,0.000000,0.356559,0.000000,0.500000
3,0,request for payment 55558,7,2,0,0.438743,0.327485,0.550000,0.664286,0.400000,...,0.954571,0.400000,0.126000,0.250000,0.002000,0.428571,0.000000,0.401620,0.000000,0.500000
4,0,request for payment 45517,8,2,2,0.503012,0.293523,0.712500,0.650000,0.440909,...,1.041264,0.454545,0.158147,0.250000,0.066294,0.428571,0.000000,0.427665,0.000000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,44,request for payment 58896,11,9,2,0.427959,0.447584,0.408333,0.535000,0.520408,...,0.425864,0.000000,0.083342,0.166667,0.000017,0.300000,0.042522,0.857470,0.000000,0.888889
312,44,request for payment 34330,11,9,2,0.439535,0.362404,0.516667,0.570000,0.563265,...,0.379927,0.000000,0.179927,0.166667,0.193187,0.200000,0.000000,0.853426,0.000000,0.888889
313,44,request for payment 58960,11,9,2,0.482208,0.339416,0.625000,0.740000,0.306122,...,0.613941,0.000000,0.135657,0.166667,0.104648,0.400000,0.078284,0.896449,0.000000,0.888889
314,44,request for payment 38995,11,9,2,0.320054,0.281775,0.358333,0.450000,0.605102,...,1.369535,0.408163,0.188348,0.166667,0.210030,0.300000,0.473023,0.936384,0.444444,1.000000


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()